# Chapter 2: Working with Text Data

Import the necessary libraries:


In [19]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.4.0
tiktoken version: 0.7.0


In [20]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [21]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [22]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [23]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [24]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [25]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [26]:
print(len(preprocessed))

4690


## 2.3 Converting tokens into token IDs

In [27]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [28]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [29]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [30]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [31]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [15]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [32]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [33]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

In [34]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [35]:
len(vocab.items())


1132

In [36]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [37]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [38]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [39]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [40]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 BytePair encoding

### What is BytePair Encoding (BPE)?

**BytePair Encoding (BPE)** is a data compression technique that has become the standard tokenization method for modern language models like GPT, BERT, and many others. It was originally developed for text compression but has proven highly effective for natural language processing tasks.

#### How BPE Works:

1. **Start with Characters**: Begin with a vocabulary of individual characters
2. **Find Most Frequent Pairs**: Identify the most frequently occurring pair of adjacent symbols
3. **Merge and Replace**: Create a new symbol representing this pair and replace all occurrences
4. **Repeat**: Continue this process until reaching a desired vocabulary size

#### Example BPE Process:
```
Initial text: "hello hello hello world world"
Initial vocab: ['h', 'e', 'l', 'o', ' ', 'w', 'r', 'd']

Step 1: Most frequent pair is "ll" → merge to "ll"
Step 2: Most frequent pair is "he" → merge to "he" 
Step 3: Most frequent pair is "ell" → merge to "ell"
...and so on
```

#### Advantages of BPE:

- **Handles Unknown Words**: Can represent any word by breaking it into subword units
- **Efficient Vocabulary**: Balances between character-level (large sequences) and word-level (huge vocabulary)
- **Language Agnostic**: Works across different languages and scripts
- **Captures Morphology**: Naturally learns prefixes, suffixes, and word roots

#### Why Modern LLMs Use BPE:

1. **Vocabulary Efficiency**: Typical BPE vocabularies are 30K-50K tokens vs millions for word-level
2. **Out-of-Vocabulary Robustness**: Can handle new words, typos, and rare terms
3. **Compression**: Reduces sequence length compared to character-level tokenization
4. **Cross-lingual Support**: Single tokenizer works across multiple languages

In [55]:
# Demonstrating BPE in action with tiktoken
import tiktoken

# Get the GPT-2 BPE tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Example 1: How BPE handles common words vs rare words
common_word = "hello"
rare_word = "antidisestablishmentarianism"
made_up_word = "supercalifragilisticexpialidocious"

print("=== BPE Tokenization Examples ===")
print(f"Common word '{common_word}':")
print(f"  Tokens: {tokenizer.encode(common_word)}")
print(f"  Decoded: {[tokenizer.decode([token]) for token in tokenizer.encode(common_word)]}")

print(f"\nRare word '{rare_word}':")
print(f"  Tokens: {tokenizer.encode(rare_word)}")
print(f"  Decoded: {[tokenizer.decode([token]) for token in tokenizer.encode(rare_word)]}")

print(f"\nMade-up word '{made_up_word}':")
print(f"  Tokens: {tokenizer.encode(made_up_word)}")
print(f"  Decoded: {[tokenizer.decode([token]) for token in tokenizer.encode(made_up_word)]}")

print(f"\nVocabulary size: {tokenizer.n_vocab:,} tokens")

=== BPE Tokenization Examples ===
Common word 'hello':
  Tokens: [31373]
  Decoded: ['hello']

Rare word 'antidisestablishmentarianism':
  Tokens: [415, 29207, 44390, 3699, 1042]
  Decoded: ['ant', 'idis', 'establishment', 'arian', 'ism']

Made-up word 'supercalifragilisticexpialidocious':
  Tokens: [16668, 9948, 361, 22562, 346, 396, 501, 42372, 498, 312, 32346]
  Decoded: ['super', 'cal', 'if', 'rag', 'il', 'ist', 'ice', 'xp', 'ial', 'id', 'ocious']

Vocabulary size: 50,257 tokens


In [56]:
# Example 2: BPE efficiency comparison
text_samples = [
    "The quick brown fox jumps over the lazy dog.",
    "Python programming language",
    "artificial intelligence machine learning",
    "COVID-19 pandemic global health crisis",
    "🌟 Hello! How are you? 😊",
    "JavaScript, HTML, CSS, and React.js"
]

print("=== BPE Efficiency Analysis ===")
for text in text_samples:
    tokens = tokenizer.encode(text)
    token_breakdown = [tokenizer.decode([token]) for token in tokens]
    
    print(f"\nText: '{text}'")
    print(f"  Character count: {len(text)}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Compression ratio: {len(text)/len(tokens):.2f} chars/token")
    print(f"  Token breakdown: {token_breakdown}")

=== BPE Efficiency Analysis ===

Text: 'The quick brown fox jumps over the lazy dog.'
  Character count: 44
  Token count: 10
  Compression ratio: 4.40 chars/token
  Token breakdown: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']

Text: 'Python programming language'
  Character count: 27
  Token count: 3
  Compression ratio: 9.00 chars/token
  Token breakdown: ['Python', ' programming', ' language']

Text: 'artificial intelligence machine learning'
  Character count: 40
  Token count: 5
  Compression ratio: 8.00 chars/token
  Token breakdown: ['art', 'ificial', ' intelligence', ' machine', ' learning']

Text: 'COVID-19 pandemic global health crisis'
  Character count: 38
  Token count: 9
  Compression ratio: 4.22 chars/token
  Token breakdown: ['CO', 'VID', '-', '19', ' pand', 'emic', ' global', ' health', ' crisis']

Text: '🌟 Hello! How are you? 😊'
  Character count: 23
  Token count: 11
  Compression ratio: 2.09 chars/token
  Token breakdown: ['�

In [57]:
# Example 3: Comparing tokenization approaches
sample_text = "The word 'unbelievable' demonstrates subword tokenization."

print("=== Tokenization Approach Comparison ===")
print(f"Sample text: '{sample_text}'")

# Character-level tokenization
char_tokens = list(sample_text)
print(f"\n1. Character-level:")
print(f"   Tokens: {len(char_tokens)}")
print(f"   First 20: {char_tokens[:20]}")

# Word-level tokenization (simple split)
word_tokens = sample_text.split()
print(f"\n2. Word-level:")
print(f"   Tokens: {len(word_tokens)}")
print(f"   Tokens: {word_tokens}")

# BPE tokenization
bpe_tokens = tokenizer.encode(sample_text)
bpe_decoded = [tokenizer.decode([token]) for token in bpe_tokens]
print(f"\n3. BPE (subword):")
print(f"   Tokens: {len(bpe_tokens)}")
print(f"   Token IDs: {bpe_tokens}")
print(f"   Decoded tokens: {bpe_decoded}")

print(f"\n=== Efficiency Summary ===")
print(f"Character-level: {len(char_tokens)} tokens")
print(f"Word-level: {len(word_tokens)} tokens") 
print(f"BPE: {len(bpe_tokens)} tokens")
print(f"\nBPE provides the best balance between vocabulary size and sequence length!")

=== Tokenization Approach Comparison ===
Sample text: 'The word 'unbelievable' demonstrates subword tokenization.'

1. Character-level:
   Tokens: 58
   First 20: ['T', 'h', 'e', ' ', 'w', 'o', 'r', 'd', ' ', "'", 'u', 'n', 'b', 'e', 'l', 'i', 'e', 'v', 'a', 'b']

2. Word-level:
   Tokens: 6
   Tokens: ['The', 'word', "'unbelievable'", 'demonstrates', 'subword', 'tokenization.']

3. BPE (subword):
   Tokens: 14
   Token IDs: [464, 1573, 705, 403, 6667, 11203, 540, 6, 15687, 850, 4775, 11241, 1634, 13]
   Decoded tokens: ['The', ' word', " '", 'un', 'bel', 'iev', 'able', "'", ' demonstrates', ' sub', 'word', ' token', 'ization', '.']

=== Efficiency Summary ===
Character-level: 58 tokens
Word-level: 6 tokens
BPE: 14 tokens

BPE provides the best balance between vocabulary size and sequence length!


In [58]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.7.0


In [59]:
tokenizer = tiktoken.get_encoding("gpt2")

In [60]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [61]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## Excercise 2.1

In [62]:
note ="teslim is a developing himself"
integers1 = tokenizer.encode(note, allowed_special={"<|endoftext|>"})

print(integers1)

[4879, 2475, 318, 257, 5922, 2241]


In [63]:
strings = tokenizer.decode(integers1)

print(strings)

teslim is a developing himself


## 2.6 Data sampling with a sliding window

In [47]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [48]:
enc_sample = enc_text[50:]

In [49]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [50]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


This code demonstrates a fundamental concept in sequence modeling and machine learning called creating input-target pairs from a sequence. This pattern is commonly used when training models to predict the next element in a sequence, such as in language modeling or time series prediction.

The code starts by setting context_size = 4, which defines how many elements will be used as input context. The variable x is created by slicing the first 4 elements from enc_sample using [:context_size]. This represents the input sequence that the model will use to make predictions.

The target sequence y is created with a crucial one-element shift: enc_sample[1:context_size+1]. This slice starts from index 1 and goes to index 5 (exclusive), giving us elements at positions 1, 2, 3, and 4. This creates a "shifted" version of the input where each element in y is the element that should come after the corresponding position in x.

The key insight is the offset relationship between x and y. If we visualize this with example data [a, b, c, d, e, f, ...]:

x would be [a, b, c, d] (positions 0-3)
y would be [b, c, d, e] (positions 1-4)
This creates training pairs where the model learns: "given a, predict b", "given b, predict c", and so on. This sliding window approach allows a single sequence to generate multiple training examples, which is essential for teaching models to understand sequential patterns and make next-token predictions.

The print statements with careful spacing show this alignment visually, making it easy to see how each input element corresponds to its target prediction.

In [54]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


install and import PyTorch

In [65]:
import torch
print("PyTorch version:", torch.__version__)
# Expected version: 2.4.0

PyTorch version: 2.4.0


In [69]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### Understanding the GPTDatasetV1 Class

This code implements a **PyTorch Dataset class** specifically designed for training GPT-style language models. The `GPTDatasetV1` class transforms raw text into the structured input-target pairs that transformer models need for next-token prediction training.

The constructor takes four key parameters: the raw text (`txt`), a tokenizer for converting text to tokens, `max_length` which defines the context window size, and `stride` which controls how much the sliding window advances between samples. The class begins by tokenizing the entire input text using the provided tokenizer, with special handling for the `<|endoftext|>` token that marks document boundaries.

**The sliding window mechanism** is the core innovation here. Rather than simply splitting text into non-overlapping chunks, this approach creates overlapping sequences by advancing the window by `stride` positions each time. For each window position `i`, it extracts an `input_chunk` of `max_length` tokens starting at position `i`, and a corresponding `target_chunk` that starts one position later (at `i + 1`). This creates the essential offset relationship where each input sequence maps to its "next token" sequence.

The input and target chunks are converted to PyTorch tensors and stored in lists. This tensor conversion is crucial because PyTorch models expect numerical tensor inputs rather than raw token lists. The `__len__` and `__getitem__` methods implement the standard PyTorch Dataset interface, allowing this class to work seamlessly with PyTorch's DataLoader for batching and iteration during training.

**A key insight** is how the stride parameter affects data utilization. A smaller stride creates more overlapping training examples (better data utilization but more computation), while a larger stride creates fewer, less overlapping examples (faster processing but potentially less effective training). This design gives practitioners fine-grained control over the trade-off between training efficiency and data coverage, which is essential for training large language models effectively.

In [70]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

### Understanding the create_dataloader_v1 Function

This function serves as a **convenient factory method** for creating PyTorch DataLoaders specifically optimized for GPT-style language model training. It abstracts away the complexity of setting up the data pipeline by combining tokenization, dataset creation, and DataLoader configuration into a single, easy-to-use interface.

The function takes several key parameters that control different aspects of the data loading process. The `txt` parameter is the raw text input, while `batch_size=4` determines how many sequences are processed together in each training step. The `max_length=256` sets the context window size (how many tokens each training sequence contains), and `stride=128` controls the overlap between consecutive training samples - a stride of 128 with a max_length of 256 means each new sample starts 128 tokens after the previous one, creating 50% overlap.

**The tokenization step** uses the GPT-2 tokenizer from tiktoken, which implements Byte Pair Encoding (BPE). This choice is crucial because it ensures compatibility with pre-trained GPT models and provides robust handling of out-of-vocabulary words. The tokenizer is initialized once and passed to the dataset, making the process efficient.

The DataLoader configuration includes several important training optimizations. `shuffle=True` randomizes the order of training samples, which helps prevent the model from learning spurious patterns based on data order. `drop_last=True` ensures all batches have exactly the same size by discarding the final incomplete batch, which is important for stable training dynamics. The `num_workers=0` parameter controls parallel data loading - setting it to 0 means data loading happens in the main process, while higher values enable multi-process loading for better performance with large datasets.

**This design pattern** exemplifies good software engineering practices by separating concerns: the function handles the "what" (creating a DataLoader), while the underlying `GPTDatasetV1` class handles the "how" (implementing the sliding window logic). This makes the code more maintainable and allows users to easily experiment with different DataLoader configurations without modifying the core dataset logic.

## 2.6 Data sampling with a sliding window

In [71]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [72]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


### Understanding the DataLoader Execution Example

This code demonstrates how to **instantiate and use the data pipeline** we've built for GPT training. Let's break down each step and understand what's happening behind the scenes.

**Step 1: Creating the DataLoader**
```python
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
```

Here we're calling our factory function with specific parameters chosen for **demonstration purposes**:
- `batch_size=1`: We want to see individual training examples, not batched ones
- `max_length=4`: Very small context window to make the output easy to understand
- `stride=1`: Move the sliding window by just 1 token each time (maximum overlap)
- `shuffle=False`: Keep the examples in their original order so we can see the sequential pattern

**Step 2: Creating an Iterator**
```python
data_iter = iter(dataloader)
```

This converts the DataLoader into an iterator object. DataLoaders in PyTorch are iterable, meaning you can loop through them, but creating an explicit iterator allows us to manually control when we fetch the next batch using `next()`.

**Step 3: Fetching the First Batch**
```python
first_batch = next(data_iter)
```

This retrieves the very first training example from our DataLoader. Since we set `batch_size=1`, this "batch" contains exactly one input-target pair.

**What the Output Shows**

The printed result will be a tuple containing two tensors:
1. **Input tensor**: The first 4 tokens from our text (positions 0-3)
2. **Target tensor**: The next 4 tokens (positions 1-4)

This creates the fundamental training pattern: given tokens [0,1,2,3], predict tokens [1,2,3,4]. The model learns to predict each next token in the sequence, which is the core of language modeling.

**Why These Parameters Matter**

The small `max_length=4` and `stride=1` settings create maximum data utilization - every possible 4-token sequence becomes a training example. In real GPT training, you'd typically use `max_length=1024` or higher and `stride=512` to balance between data efficiency and computational cost. The `shuffle=False` setting lets us see the natural progression of the sliding window through the text.

In [73]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


## Excercise 2.2

Exercise 2.2 Data loaders with different strides and context sizes

To develop more intuition for how the data loader works, try to run it with different

settings such as max_length=2 and stride=2, and max_length=8 and stride=2.

In [74]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)

data_iter = iter(dataloader)
third_batch = next(data_iter)
print(third_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]


In [75]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False
)

data_iter = iter(dataloader)
fourth_batch = next(data_iter)
print(fourth_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


### How Stride Affects Memory Usage and Data Overlap

Excellent question! The relationship between stride and memory usage is **counterintuitive** - let me explain what's really happening:

#### The Stride-Memory Relationship

**Smaller stride = MORE memory usage (more training samples)**
**Larger stride = LESS memory usage (fewer training samples)**

Here's why:

#### Example with max_length=8 and different strides:

**Scenario 1: stride=1 (maximum overlap)**
```
Sample 1: tokens [0,1,2,3,4,5,6,7] → target [1,2,3,4,5,6,7,8]
Sample 2: tokens [1,2,3,4,5,6,7,8] → target [2,3,4,5,6,7,8,9]
Sample 3: tokens [2,3,4,5,6,7,8,9] → target [3,4,5,6,7,8,9,10]
...and so on
```
- **High overlap**: Each token appears in ~8 different training samples
- **More samples**: From 1000 tokens, you get ~992 training examples
- **Higher memory**: Need to store all these overlapping samples

**Scenario 2: stride=8 (no overlap)**
```
Sample 1: tokens [0,1,2,3,4,5,6,7] → target [1,2,3,4,5,6,7,8]
Sample 2: tokens [8,9,10,11,12,13,14,15] → target [9,10,11,12,13,14,15,16]
Sample 3: tokens [16,17,18,19,20,21,22,23] → target [17,18,19,20,21,22,23,24]
...and so on
```
- **No overlap**: Each token appears in exactly 1 training sample
- **Fewer samples**: From 1000 tokens, you get ~125 training examples
- **Lower memory**: Much fewer samples to store

#### The Trade-off

- **Small stride (like stride=1)**: Maximum data utilization but higher memory usage
- **Large stride (like stride=max_length)**: Minimal memory usage but potential information loss
- **Medium stride (like stride=max_length/2)**: Balanced approach - some overlap with reasonable memory usage

#### Why This Matters for GPT Training

In our example with `max_length=8, stride=2`:
- We get 50% overlap between consecutive samples
- Each token participates in multiple training contexts
- This helps the model learn better representations while keeping memory manageable
- It's a sweet spot between data efficiency and computational cost

So you're right that stride affects memory, but **smaller strides actually use MORE memory** because they create more training samples with greater overlap!

In [76]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Question: "Can I say that batch_size is column and max_length is row?"

### Understanding Batch Dimensions: Rows vs Columns

**Excellent observation!** Yes, you can absolutely think of it that way:

- **`batch_size` = number of rows** (how many training examples)
- **`max_length` = number of columns** (how many tokens per example)

#### Tensor Shape Visualization

When we set `batch_size=8` and `max_length=4`, our tensors have shape `[8, 4]`:

```
        Token 1  Token 2  Token 3  Token 4  ← max_length (columns)
Row 1:  [  15,     24,     31,     42  ]   ← Sample 1
Row 2:  [  24,     31,     42,     58  ]   ← Sample 2  
Row 3:  [  31,     42,     58,     71  ]   ← Sample 3
Row 4:  [  42,     58,     71,     89  ]   ← Sample 4
Row 5:  [  58,     71,     89,     95  ]   ← Sample 5
Row 6:  [  71,     89,     95,    103  ]   ← Sample 6
Row 7:  [  89,     95,    103,    112  ]   ← Sample 7
Row 8:  [  95,    103,    112,    124  ]   ← Sample 8
↑
batch_size (rows)
```

#### Why This Matrix Structure Matters:

**Each Row = One Training Example**
- Row 1 contains the first 4 tokens of the sequence
- Row 2 contains tokens 2-5 (shifted by stride=4, so no overlap in this case)
- Each row is processed independently by the model

**Each Column = Token Position**
- Column 1: The first token in each training sequence
- Column 2: The second token in each training sequence
- Column 3: The third token in each training sequence
- Column 4: The fourth token in each training sequence

#### PyTorch Convention:

In PyTorch, this is typically written as `[batch_size, sequence_length]` or `[B, L]`:
- **B (batch dimension)**: How many examples we process simultaneously
- **L (length dimension)**: How many tokens each example contains

#### Why Batching Matters:

Processing 8 examples simultaneously (batch_size=8) is much more efficient than processing them one by one, because:
1. **GPU Parallelization**: Modern GPUs excel at parallel matrix operations
2. **Memory Efficiency**: Loading multiple examples at once reduces memory overhead
3. **Training Stability**: Gradients computed over multiple examples are more stable than single-example gradients

So yes, your mental model is perfect: **batch_size = rows, max_length = columns**!

## 2.7 Creating token embeddings

## 2.8 Encoding word positions